%md
# 01 — Profile Raw Data (Incremental Bronze)
Profiles one Bronze batch from ADLS Gen2. Missing source folders are allowed.
No data is modified or written by this notebook.


In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("batch_id", "")
batch_id = dbutils.widgets.get("batch_id").strip()

if not batch_id:
    raise ValueError("batch_id is required")

BASE_PATH = "abfss://edtech@edtechpipline26.dfs.core.windows.net"
RAW_PATH = f"{BASE_PATH}/raw/batch_id={batch_id}"

print(f"Profiling Bronze batch: {batch_id}")
print(f"Path: {RAW_PATH}")


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-4898083646566382>, line 7
      4 batch_id = dbutils.widgets.get("batch_id").strip()
      6 if not batch_id:
----> 7     raise ValueError("batch_id is required")
      9 BASE_PATH = "abfss://edtech@edtechpipline26.dfs.core.windows.net"
     10 RAW_PATH = f"{BASE_PATH}/raw/batch_id={batch_id}"

ValueError: batch_id is required

%md
## Discover available sources


In [0]:
try:
    available_sources = {
        item.name.rstrip("/")
        for item in dbutils.fs.ls(RAW_PATH)
        if item.isDir()
    }
except Exception as e:
    raise ValueError(
        f"Batch path does not exist or cannot be read: {RAW_PATH}"
    ) from e

supported_sources = {
    "devto",
    "freecodecamp",
    "geeksforgeeks",
    "medium",
    "pluralsight",
}

available_sources = available_sources.intersection(supported_sources)

if not available_sources:
    raise ValueError(f"No supported source folders found in batch {batch_id}")

print("Available sources:", ", ".join(sorted(available_sources)))


%md
## Read available raw sources


In [0]:
dataframes = {}

if "devto" in available_sources:
    dataframes["dev.to"] = (
        spark.read.option("multiLine", True)
        .json(f"{RAW_PATH}/devto/*.json")
    )

if "freecodecamp" in available_sources:
    dataframes["freeCodeCamp"] = (
        spark.read.option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{RAW_PATH}/freecodecamp/*.csv")
    )

if "geeksforgeeks" in available_sources:
    dataframes["GeeksforGeeks"] = (
        spark.read.option("multiLine", True)
        .json(f"{RAW_PATH}/geeksforgeeks/*.json")
    )

if "medium" in available_sources:
    dataframes["Medium"] = (
        spark.read.option("multiLine", True)
        .json(f"{RAW_PATH}/medium/*.json")
    )

if "pluralsight" in available_sources:
    dataframes["Pluralsight"] = (
        spark.read.option("multiLine", True)
        .json(f"{RAW_PATH}/pluralsight/*.json")
    )


%md
## Row and column counts


In [0]:
summary = []

for name, df in dataframes.items():
    summary.append((name, df.count(), len(df.columns)))

display(
    spark.createDataFrame(
        summary,
        ["source", "row_count", "column_count"]
    ).orderBy(F.desc("row_count"))
)


%md
## Schemas


In [0]:
for name, df in dataframes.items():
    print(f"\n{'=' * 60}\n{name.upper()}\n{'=' * 60}")
    df.printSchema()


%md
## Sample records


In [0]:
for name, df in dataframes.items():
    print(f"\n===== {name} =====")

    preview = [
        c for c in [
            "source",
            "category",
            "topic",
            "title",
            "author",
            "publication_date",
            "published_at",
            "url",
        ]
        if c in df.columns
    ]

    display(df.select(*preview).limit(5))


%md
## Duplicate URLs within this batch


In [0]:
duplicate_summary = []

for name, df in dataframes.items():
    if "url" in df.columns:
        dup_count = (
            df.filter(
                F.col("url").isNotNull()
                & (F.trim(F.col("url")) != "")
            )
            .groupBy("url")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )
    else:
        dup_count = 0

    duplicate_summary.append((name, dup_count))

display(
    spark.createDataFrame(
        duplicate_summary,
        ["source", "duplicate_url_groups"]
    )
)


%md
## Missing-value summary for key fields


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)

quality_rows = []

for name, df in dataframes.items():
    total = df.count()

    def missing_count(column_name):
        if column_name not in df.columns:
            return None

        return df.filter(
            F.col(column_name).isNull()
            | (F.trim(F.col(column_name).cast("string")) == "")
        ).count()

    quality_rows.append((
        name,
        total,
        missing_count("title"),
        missing_count("url"),
        missing_count("content"),
    ))

# Explicit schema prevents CANNOT_DETERMINE_TYPE
# when an incremental batch contains only some sources
# and a column contains only None values.
quality_schema = StructType([
    StructField("source", StringType(), False),
    StructField("row_count", LongType(), True),
    StructField("missing_title", LongType(), True),
    StructField("missing_url", LongType(), True),
    StructField("missing_content", LongType(), True),
])

quality_df = spark.createDataFrame(
    quality_rows,
    schema=quality_schema,
)

display(quality_df)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4898083646566396>, line 10
      1 from pyspark.sql.types import (
      2     StructType,
      3     StructField,
      4     StringType,
      5     LongType,
      6 )
      8 quality_rows = []
---> 10 for name, df in dataframes.items():
     11     total = df.count()
     13     def missing_count(column_name):

NameError: name 'dataframes' is not defined